In [ ]:
# --- Setup & Imports ---
import os
import sys

# Clone repository
REPO_URL = "https://github.com/ThaiLearnCoding/Coco_classification.git"
REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')

if not os.path.exists(REPO_NAME):
    print(f"Cloning repository: {REPO_URL}")
    !git clone {REPO_URL}

# Move into the repository
if os.path.exists(REPO_NAME):
    os.chdir(REPO_NAME)

# Install dependencies
print("Installing requirements...")
!pip install -q -r requirements.txt
!pip install -q gdown # Install gdown for Google Drive downloads

In [ ]:
# --- Add utilities ---

# Standard imports
import json
import yaml
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from zipfile import ZipFile
from collections import Counter
import numpy as np
from torch.utils.data import DataLoader, Subset

# Add src to path so we can import modules
# Assuming notebook is in 'notebooks/' and src is in '../src/'
if os.getcwd().endswith('notebooks'):
    sys.path.append('../src')
    os.chdir('..') # Change CWD to project root for data/config access
else:
    sys.path.append('./src')

# Import project modules
try:
    from models import CLIPFewShotModel
    from data_utils import get_dataloaders, CocoMultimodalDataset
    from engine import evaluate_zero_shot, train_few_shot
except ImportError as e:
    print(f"Error importing modules: {e}")
    print("Make sure you are in the project root and 'src' directory exists.")

print(f"Current Working Directory: {os.getcwd()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

In [ ]:
# --- Exploratory Data Analysis (EDA) ---

if not os.path.exists('data'):
    os.makedirs('data')

# Extract file ID from the Google Drive sharing link
zip_file_id = '1VtF-EVa53F6HezvCUWz6qQMueZOoRDPR'
zip_path_colab = 'data/coco_multimodal_subset.zip'

# Download the file using gdown if it doesn't already exist
if not os.path.exists(zip_path_colab):
    print(f"Downloading data from Google Drive (ID: {zip_file_id})...")
    !gdown --id {zip_file_id} -O {zip_path_colab}
else:
    print(f"Zip file already exists at {zip_path_colab}.")

ZIP_PATH = zip_path_colab # Ensure ZIP_PATH is correctly set for subsequent use

# Read metadata to visualize Class Distribution
if os.path.exists(ZIP_PATH):
    with ZipFile(ZIP_PATH, 'r') as z:
        if 'metadata.json' in z.namelist():
            metadata = json.loads(z.read('metadata.json'))

            # Count labels
            all_labels = [label for item in metadata for label in item['labels']]
            label_counts = Counter(all_labels)

            # Plot
            plt.figure(figsize=(10, 5))
            plt.bar(label_counts.keys(), label_counts.values())
            plt.title('Class Distribution in COCO Subset')
            plt.xlabel('Classes')
            plt.ylabel('Number of Images')
            plt.xticks(rotation=45)
            plt.show()
        else:
            print("metadata.json not found in zip.")
else:
    print("Cannot perform EDA: Zip file not found after download attempt.")

In [ ]:
# --- Configuration & Model Initialization ---

# Load Config
CONFIG_PATH = 'configs/config.yaml'

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
# print(yaml.dump(config, sort_keys=False))

device = config['model']['device'] if torch.cuda.is_available() else 'cpu'

# Initialize Model
# Uses CLIP backbone (frozen) + Linear Classifier
model = CLIPFewShotModel(config['model']['name'], len(config['data']['classes']), device)
print(f"Model initialized: {config['model']['name']}")

In [ ]:
# --- Data Loading ---

# Get DataLoaders (Full Train / Test split from config)
# Note: full_train_loader contains 80% of data, used for sampling k-shots later
full_train_loader, test_loader = get_dataloaders(config, model.preprocess)

print(f"Full Train size: {len(full_train_loader.dataset)}")
print(f"Test size: {len(test_loader.dataset)}")

In [ ]:
# --- Zero-Shot Evaluation ---
# Evaluate CLIP's zero-shot capability before any training

print("Running Zero-Shot Evaluation...")
zero_shot_report = evaluate_zero_shot(model, test_loader, config['data']['classes'], device)
print("\n=== Zero-Shot Classification Report ===")
print(zero_shot_report)

In [ ]:
# --- Few-Shot Sampling & Training Setup ---

def create_few_shot_subset(dataset, k=16, classes=None):
    """
    Selects k samples per class from the dataset.
    """
    if classes is None:
        return dataset

    # Access the underlying dataset if it's a Subset
    if isinstance(dataset, Subset):
        root_dataset = dataset.dataset
        subset_indices = dataset.indices
    else:
        root_dataset = dataset
        subset_indices = range(len(dataset))

    class_indices = {cls: [] for cls in classes}

    # Iterate through the available indices in the training subset
    for idx in subset_indices:
        # Access metadata directly from root dataset
        meta = root_dataset.filtered_data[idx]

        # Find which class this sample belongs to (first matching target class)
        label_name = [l for l in meta['labels'] if l in classes]
        if label_name:
            label_name = label_name[0]
            if len(class_indices[label_name]) < k:
                class_indices[label_name].append(idx) # Keep original index

    # Collect all selected indices
    final_indices = []
    for cls, indices in class_indices.items():
        if len(indices) < k:
            print(f"Warning: Class '{cls}' has only {len(indices)} samples, requested {k}.")
        final_indices.extend(indices)

    return Subset(root_dataset, final_indices)

def reset_model(config):
    """Re-initialize the model to reset weights for fair comparison."""
    return CLIPFewShotModel(config['model']['name'], len(config['data']['classes']), config['model']['device'])

# Add this utility to get accuracy
from sklearn.metrics import accuracy_score

def evaluate_accuracy(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return accuracy_score(all_labels, all_preds)

print("Evaluation utilities ready.")

In [ ]:
# --- Final Evaluation (after Few-Shot) ---

print("Running Evaluation after Few-Shot Training...")
final_report = evaluate_zero_shot(model, test_loader, config['data']['classes'], device)
print("\n=== Few-Shot Classification Report ===")
print(final_report)

In [ ]:
# --- Compare One-Shot vs Few-Shot Performance ---

k_values = [1, 5, 8, 16] # Define K-shots to compare (1 = One-shot)
accuracies = []

print("Evaluating Zero-shot Baseline..."):
base_model = CLIPFewShotModel(config['model']['name'], len(config['data']['classes']), device)
report_dict = evaluate_zero_shot(base_model, test_loader, config['data']['classes'], device)

print(f"\n--- Starting Comparison Experiment: K={k_values} ---")

for k in k_values:
    print(f"\nTraining with K={k} shots per class...")

    # Reset Model
    model = reset_model(config)
    optimizer = optim.Adam(model.classifier.parameters(), lr=config['train']['learning_rate'])
    criterion = nn.CrossEntropyLoss()

    # Prepare Data
    few_shot_dataset = create_few_shot_subset(full_train_loader.dataset, k=k, classes=config['data']['classes'])
    few_shot_loader = DataLoader(few_shot_dataset, batch_size=config['data']['batch_size'], shuffle=True)

    # Train
    # Suppress output for loop cleanliness if desired, or keep it
    train_few_shot(model, few_shot_loader, criterion, optimizer, config['train']['epochs'], device)

    # Evaluate
    acc = evaluate_accuracy(model, test_loader, device)
    accuracies.append(acc)
    print(f"--> K={k} Accuracy: {acc:.4f}")

# Plot Results
plt.figure(figsize=(10, 6))
plt.plot(k_values, accuracies, marker='o', linestyle='-', color='b')
plt.title('Few-Shot Classification Performance (CLIP Linear Probe)')
plt.xlabel('Number of Shots (K)')
plt.ylabel('Top-1 Accuracy')
plt.grid(True)
plt.xticks(k_values)
for i, txt in enumerate(accuracies):
    plt.annotate(f"{txt:.2f}", (k_values[i], accuracies[i]), textcoords="offset points", xytext=(0,10), ha='center')
plt.show()